In [1]:
import os 

In [2]:
os.chdir("..\.")

In [3]:
import torch 
import torch.nn.functional as F

In [4]:
torch.manual_seed(1337)
with open ("data\gpt_train.txt","r") as file:
    text    = file.read()


# All the unique characters that occur in this text 
chars       = sorted((set(text)))
vocab_size  = len(chars)

## creating mapping from characters to integers 
stoi    = {ch:i for i,ch in enumerate(chars)}
itos    = dict(enumerate(chars))
encode  = lambda word: [stoi[i] for i in word]
decode  = lambda integers: "".join(itos[int(i)] for i in integers)



## Let's encode entire dataset of file. 
data = torch.tensor(encode(text),dtype=torch.long,device="cuda")

## Lets split the data into train and val dataset 
n   = int(0.9 * len(data))
train_data  = data[:n]
val_data    = data[n:]

## FeedForward Layer 

- A feedforward layer in a Transformer block is not just a single layer but typically a small neural network itself, consisting of two linear transformations (fully connected layers) with a non-linear activation function (usually ReLU or GELU) in between.
- Another word: adding comutation into network. 


**In Multi-head self-attention layer they just did the communication between the tokens, but they forgot to think on what they foung from the other tokens.*

### The key characteristics are:

**1. Position-wise Application:** 
    - This is a critical distinction. Unlike the multi-head attention mechanism which considers the entire sequence to compute contextual representations, the FFN is applied independently and identically to each position (e.g., each word's vector representation) in the sequence.
    - If you have a sequence of N tokens, each token's vector representation passes through the same FFN, but the computations for each token are independent of the others. This parallelism is a significant advantage of Transformers.

**2. Expansion and Contraction:**

- The first linear layer typically expands the dimensionality of the input vector.
    - Example:  if the input vector for a token has a dimension of dmodel​ (e.g., 512), the first linear layer might project it to a much larger dimension, say dff​ (e.g., 2048 ==> 4*512 ).
- The non-linear activation function is applied to this expanded representation.
- The second linear layer then contracts this expanded representation back to the original dmodel​ dimensionality.

- Mathematically, for an input x to the FFN (which is a vector representing a single token's output from the attention sublayer):
    - FFN(x)=max(0,xW1​+b1​)W2​+b2​

- where W1​,b1​,W2​,b2​ are learnable weight matrices and bias vectors, and max(0,⋅) is the ReLU activation function (though GELU is often used in modern Transformers).


**3. Residual Connection and Layer Normalization:**

- Like the self-attention sub-layer, the FFN sub-layer in a Transformer is usually followed by a residual connection (adding the input of the sub-layer to its output) and then layer normalization. This helps with training stability and allows for deeper networks.

In [16]:
import torch
from tqdm import tqdm
from torch import nn
from torch.nn import functional as F
from common.data_processing import get_batch     


batch_size  = 4     # B 
block_size  = 8     # T
n_emd       = 32    # C 
device      = "cuda" if torch.cuda.is_available() else "cpu"
eval_interval   = 300
learning_rate   = 1e-3 
max_iters       = 5000 
eval_iter       = 200
n_layers        = 6 
n_heads         = 4
dropout         = 0.2

vocab_size      = 65 

In [17]:
class Head(nn.Module):
    def __init__(self,head_size):
        super().__init__()
        self.key    = nn.Linear(n_emd,head_size,bias=False)   # key projection
        self.query  = nn.Linear(n_emd,head_size,bias=False)   # query projection
        self.value  = nn.Linear(n_emd,head_size,bias=False)   # value projection 
        self.register_buffer('tril',torch.tril(torch.ones(block_size,block_size)))
    def forward(self,x):
        _,T,C   = x.shape
        k       = self.key(x)   # (B,T,head_size) ==> 4,8,16
        q       = self.query(x) # (B,T,head_size) ==> 4,8,16   
        ## compute attention score(affinities)
        wei     = q @ k.transpose(-2,-1) * C ** -0.5   # (B,T,c) @ (B,C,T) ==> (B,T,T)
        wei     = wei.masked_fill(self.tril[:T,:T] == 0, float('-inf'))
        wei     = F.softmax(wei,dim=-1)
        # perform the weighted aggregation of the value
        v       = self.value(x) # (B,T,head_size) ==> 4,8,16
        out     = wei @ v       # (B,T,T) @ (B,T,head_size) ==> (B,T,head_size) 
        return out 

In [18]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads,head_size):
        super().__init__()
        self.heads   = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
    def forward(self,x):
        return torch.cat([head(x) for head in self.heads],dim=-1)   # concatenate in the channel dimension 

In [19]:
class FeedForward(nn.Module):
    def __init__(self,n_emd):
        super().__init__()
        self.net    = nn.Sequential(
            nn.Linear(n_emd,n_emd),
            nn.ReLU()
        )
    def forward(self,x):
        return self.net(x)

In [20]:
## make a block: 
class Block(nn.Module):
    def __init__(self,n_emd,n_heads):
        super().__init__()
        head_size                   = n_emd // n_heads 
        self.self_attention_head    = MultiHeadAttention(n_heads,head_size)
        self.ffwd                   = FeedForward(n_emd)
    def forward(self,x):
        x   = self.self_attention_head(x)
        x   = self.ffwd(x)
        return x 

In [21]:
block   = Block(n_emd,n_heads=4)
x       = torch.randn(batch_size,batch_size,n_emd)
block(x).shape

torch.Size([4, 4, 32])

In [22]:
class BiGramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for next token from a lookup table 
        self.token_embedding_table      = nn.Embedding(vocab_size,n_emd)
        self.position_embedding_table   = nn.Embedding(block_size,n_emd)
        self.block                      = nn.Sequential(
            Block(n_emd,n_heads=4),
            Block(n_emd,n_heads=4),
            Block(n_emd,n_heads=4),
            Block(n_emd,n_heads=4),
        )
        self.lm_head                    = nn.Linear(n_emd,vocab_size)       
        

    def forward(self,idx,target=None):
        B,T         = idx.shape 
        # index and target are both (B,T) tensor of integers
        tok_emb     = self.token_embedding_table(idx)   # its arrange in the shape of ==================================================================>> (B,T,C)
        pos_emb     = self.position_embedding_table(torch.arange(T,device=device)) # This embdding gives the idea of where word is belong in a sentence=>> (T,C) 
        x           = tok_emb + pos_emb                 # combined representation of token and its position.======Broadcasting apply=>> (B,T,C) + (T,C) == (B,T,C) 
        x           = self.block(x)                     # 
        logits      = self.lm_head(x)                   # for getting token_emb to logits we need linear layer,shape ===================================>> (B,T,vocab_size) 

        if target is None:
            loss    = None
        else:    
            B,T,C   = logits.shape
            logits  = logits.view(B*T,C)    # cross entropy input expectation is (minibatch,C)
            target  = target.view(B*T)      
            loss    = nn.functional.cross_entropy(logits,target)
        return logits,loss
    
    def generate(self,idx,max_new_tokens):
        # idx is (B,T) array of indices in the current context. 
        for _ in range(max_new_tokens):
            idx_cond    = idx[:,-block_size:]                   # crop idx to last block_size tokens
            logits,loss = self.forward(idx_cond)                # Get the prediction ==>  (B,T,C)
            logits      = logits[:,-1,:]                        # focus only on last time step  ==>  (B,C)
            probs       = nn.functional.softmax(logits,dim=-1)  # (B,C)
            # sample from the distribution 
            idx_next    = torch.multinomial(probs,num_samples=1)    # (B,1)
            # append sample index to running sequence 
            idx         = torch.cat([idx,idx_next],dim=1)           # (B,T+1)
        return idx

In [23]:
model = BiGramLanguageModel()
model = model.to("cuda:0")

In [24]:
print(sum(p.numel() for p in model.parameters())/1e6, 'M parameters')

0.020993 M parameters


In [25]:

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ["train","val"]:
        losses = torch.zeros(eval_iter)
        for k in range(eval_iter):
            X,Y         = get_batch("train")
            logits,loss = model(X,Y)
            losses[k]   = loss.item()   
        out[split] = losses.mean()
    model.train()
    return out

In [26]:

optimizer   = torch.optim.AdamW(model.parameters(),lr=learning_rate)


for iter in range(max_iters):
    if iter % eval_interval == 0:
        losses  = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f},val loss {losses['val']:.4f}")

    #sample a batch of data
    xb,yb = get_batch("train")
    #evaluate the loss
    logits,loss = model(xb,yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()


context = torch.zeros((1,1),dtype=torch.long,device="cuda:0")
print(decode(model.generate(context,max_new_tokens=500)[0]))

step 0: train loss 4.1851,val loss 4.1845
step 300: train loss 3.3204,val loss 3.3136
step 600: train loss 3.2931,val loss 3.2772
step 900: train loss 3.2528,val loss 3.2319
step 1200: train loss 3.2150,val loss 3.2378
step 1500: train loss 3.1847,val loss 3.1714
step 1800: train loss 3.1465,val loss 3.1779
step 2100: train loss 3.1231,val loss 3.1166
step 2400: train loss 3.0206,val loss 3.0213
step 2700: train loss 3.0131,val loss 2.9819
step 3000: train loss 2.9396,val loss 2.9636
step 3300: train loss 2.9182,val loss 2.9110
step 3600: train loss 2.8793,val loss 2.9137
step 3900: train loss 2.8417,val loss 2.8514
step 4200: train loss 2.7860,val loss 2.7657
step 4500: train loss 2.7570,val loss 2.7616
step 4800: train loss 2.7415,val loss 2.7680


ILAU:

O Pa horl yhhiu a fe o, noenew hronv i a 'ide henns fes cerwnece.

Ait, hone mk po gee-muite I b arkt? crolns es o se mot, fetaks
Set thowt cosgo as hoawn, pow? pEanr le;,

U
DIYL:
Tnh vhy avls sobiec a lute tese wipe yi not mk mree

### Note:

- when block is increases the network gets deeper and deeper it cause the vanishing gradient problem." While increasing the number of blocks does make the network deeper, and deeper networks can be susceptible to vanishing (or exploding) gradients. 2 Optimization methods solve this issues. 
    - 1. Residual connection
    - 2. Layer Normalization 
- While it's not entirely immune, it's less prone to it than, say, a deep plain RNN or a very deep feed-forward network without residual connections.

# 1. Residual Connection / Skip Connections

- Its primary purpose is enable the training of much deeper network by mitigate issues like vanishing gradinet and degradation problem. 
- Its creating a "shortcut" or "bypass" path that directly feeds the input of a block of layers to its output.

- Mathematically, consider a block of layers that transforms an input x into an output H(x). In a traditional network, the output of this block would simply be H(x). With a residual connection, the output becomes:

Y=F(x)+x

Where:
    - x is the input to the residual block.

- F(x) ==> transformation performed by the stacked layers within the block (e.g., convolutional layers, activation functions, normalization). This F(x) is often referred to as the residual function or residual mapping.

The x added at the end is the identity mapping, or the "shortcut connection."

In [27]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads,head_size):
        super().__init__()
        self.heads      = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj       = nn.Linear(n_emd,n_emd)    
    def forward(self,x):
        out     = torch.cat([head(x) for head in self.heads],dim=-1)   # concatenate in the channel dimension 
        return self.proj(out)   # projection is the linear transformation of the out 

In [28]:
class FeedForward(nn.Module):
    def __init__(self,n_emd):
        super().__init__()
        self.net    = nn.Sequential(
            nn.Linear(n_emd, 4 * n_emd), # from Paper 'attention all you need'
            nn.ReLU(),   
            nn.Linear(4 * n_emd,n_emd)
        )
    def forward(self,x):
        return self.net(x)

In [29]:
## residual connection : 
class Block(nn.Module):
    def __init__(self,n_emd,n_heads):
        super().__init__()
        head_size                   = n_emd // n_heads 
        self.self_attention_head    = MultiHeadAttention(n_heads,head_size)
        self.ffwd                   = FeedForward(n_emd)
    def forward(self,x):
        x   = x + self.self_attention_head(x)   # adding x is residula connection 
        x   = self.ffwd(x)                      # adding x is residula connection 
        return x 

In [30]:
class BiGramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for next token from a lookup table 
        self.token_embedding_table      = nn.Embedding(vocab_size,n_emd)
        self.position_embedding_table   = nn.Embedding(block_size,n_emd)
        self.block                      = nn.Sequential(
            Block(n_emd,n_heads=4),
            Block(n_emd,n_heads=4),
            Block(n_emd,n_heads=4),
            Block(n_emd,n_heads=4),
        )
        self.lm_head                    = nn.Linear(n_emd,vocab_size)       
        

    def forward(self,idx,target=None):
        B,T         = idx.shape 
        # index and target are both (B,T) tensor of integers
        tok_emb     = self.token_embedding_table(idx)   # its arrange in the shape of ==================================================================>> (B,T,C)
        pos_emb     = self.position_embedding_table(torch.arange(T,device=device)) # This embdding gives the idea of where word is belong in a sentence=>> (T,C) 
        x           = tok_emb + pos_emb                 # combined representation of token and its position.======Broadcasting apply=>> (B,T,C) + (T,C) == (B,T,C) 
        x           = self.block(x)                     # 
        logits      = self.lm_head(x)                   # for getting token_emb to logits we need linear layer,shape ===================================>> (B,T,vocab_size) 

        if target is None:
            loss    = None
        else:    
            B,T,C   = logits.shape
            logits  = logits.view(B*T,C)    # cross entropy input expectation is (minibatch,C)
            target  = target.view(B*T)      
            loss    = nn.functional.cross_entropy(logits,target)
        return logits,loss
    
    def generate(self,idx,max_new_tokens):
        # idx is (B,T) array of indices in the current context. 
        for _ in range(max_new_tokens):
            idx_cond    = idx[:,-block_size:]                   # crop idx to last block_size tokens
            logits,loss = self.forward(idx_cond)                # Get the prediction ==>  (B,T,C)
            logits      = logits[:,-1,:]                        # focus only on last time step  ==>  (B,C)
            probs       = nn.functional.softmax(logits,dim=-1)  # (B,C)
            # sample from the distribution 
            idx_next    = torch.multinomial(probs,num_samples=1)    # (B,1)
            # append sample index to running sequence 
            idx         = torch.cat([idx,idx_next],dim=1)           # (B,T+1)
        return idx


In [31]:
model = BiGramLanguageModel()
model = model.to("cuda:0")

In [32]:

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ["train","val"]:
        losses = torch.zeros(eval_iter)
        for k in range(eval_iter):
            X,Y         = get_batch("train")
            logits,loss = model(X,Y)
            losses[k]   = loss.item()   
        out[split] = losses.mean()
    model.train()
    return out

In [33]:

optimizer   = torch.optim.AdamW(model.parameters(),lr=learning_rate)


for iter in range(max_iters):
    if iter % eval_interval == 0:
        losses  = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f},val loss {losses['val']:.4f}")

    #sample a batch of data
    xb,yb = get_batch("train")
    #evaluate the loss
    logits,loss = model(xb,yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()


context = torch.zeros((1,1),dtype=torch.long,device="cuda:0")
print(decode(model.generate(context,max_new_tokens=500)[0]))

step 0: train loss 4.1800,val loss 4.1801
step 300: train loss 2.9723,val loss 2.9786
step 600: train loss 2.8002,val loss 2.8075
step 900: train loss 2.6838,val loss 2.7198
step 1200: train loss 2.6259,val loss 2.6277
step 1500: train loss 2.6263,val loss 2.5634
step 1800: train loss 2.5484,val loss 2.5526
step 2100: train loss 2.5421,val loss 2.5720
step 2400: train loss 2.5339,val loss 2.5001
step 2700: train loss 2.4736,val loss 2.5007
step 3000: train loss 2.5246,val loss 2.4976
step 3300: train loss 2.4381,val loss 2.4478
step 3600: train loss 2.4361,val loss 2.4548
step 3900: train loss 2.4174,val loss 2.4509
step 4200: train loss 2.4401,val loss 2.3740
step 4500: train loss 2.4221,val loss 2.4090
step 4800: train loss 2.3690,val loss 2.3763

Whapild nowe, the se vemise wre. you rave hay d my siy ther that saner thefe me:

A'y your parnd mut ewa isis Oe nyeour the rewe hy
I sade resthy I ry lowe ny that soud lids citw of bur pornenoode
Theou bape crath dir is ie tre a
ve he ve l

In [ ]:
print(sum(p.numel() for p in model.parameters())/1e6, 'M parameters')

0.050177 M parameters


# 2. Layer Normalization 

- Normalizes the inputs across all features within a single sample (or a single layer of a single sample).

- For each individual input in the batch, LN calculates the mean and variance across all its features (e.g., all activations within a specific layer for that single image).

- #### *This means it normalizes "horizontally" across the feature dimension.*

![Architecture](Transformer_Architecture.png "Transformer Architecture")

In [34]:
## residual connection : 
class Block(nn.Module):
    def __init__(self,n_emd,n_heads):
        super().__init__()
        head_size                   = n_emd // n_heads 
        self.self_attention_head    = MultiHeadAttention(n_heads,head_size)
        self.ffwd                   = FeedForward(n_emd)
        self.ln1                    = nn.LayerNorm(n_emd)
        self.ln2                    = nn.LayerNorm(n_emd)
    def forward(self,x):
        x   = x + self.self_attention_head(self.ln1(x))   # adding x is residula connection, applying layer normalization  
        x   = self.ffwd(self.ln2(x))                      # adding x is residula connection, applying layer normalization 
        return x 

In [35]:
class BiGramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for next token from a lookup table 
        self.token_embedding_table      = nn.Embedding(vocab_size,n_emd)
        self.position_embedding_table   = nn.Embedding(block_size,n_emd)
        self.block                      = nn.Sequential(
            Block(n_emd,n_heads=4),
            Block(n_emd,n_heads=4),
            Block(n_emd,n_heads=4),
        )
        self.lm_head                    = nn.Linear(n_emd,vocab_size)       
        

    def forward(self,idx,target=None):
        B,T         = idx.shape 
        # index and target are both (B,T) tensor of integers
        tok_emb     = self.token_embedding_table(idx)   # its arrange in the shape of ==================================================================>> (B,T,C)
        pos_emb     = self.position_embedding_table(torch.arange(T,device=device)) # This embdding gives the idea of where word is belong in a sentence=>> (T,C) 
        x           = tok_emb + pos_emb                 # combined representation of token and its position.======Broadcasting apply=>> (B,T,C) + (T,C) == (B,T,C) 
        x           = self.block(x)                     # 
        logits      = self.lm_head(x)                   # for getting token_emb to logits we need linear layer,shape ===================================>> (B,T,vocab_size) 

        if target is None:
            loss    = None
        else:    
            B,T,C   = logits.shape
            logits  = logits.view(B*T,C)    # cross entropy input expectation is (minibatch,C)
            target  = target.view(B*T)      
            loss    = nn.functional.cross_entropy(logits,target)
        return logits,loss
    
    def generate(self,idx,max_new_tokens):
        # idx is (B,T) array of indices in the current context. 
        for _ in range(max_new_tokens):
            idx_cond    = idx[:,-block_size:]                   # crop idx to last block_size tokens
            logits,loss = self.forward(idx_cond)                # Get the prediction ==>  (B,T,C)
            logits      = logits[:,-1,:]                        # focus only on last time step  ==>  (B,C)
            probs       = nn.functional.softmax(logits,dim=-1)  # (B,C)
            # sample from the distribution 
            idx_next    = torch.multinomial(probs,num_samples=1)    # (B,1)
            # append sample index to running sequence 
            idx         = torch.cat([idx,idx_next],dim=1)           # (B,T+1)
        return idx


In [36]:
model = BiGramLanguageModel()
model = model.to("cuda:0")

In [37]:
print(sum(p.numel() for p in model.parameters())/1e6, 'M parameters')


0.042305 M parameters


In [38]:

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ["train","val"]:
        losses = torch.zeros(eval_iter)
        for k in range(eval_iter):
            X,Y         = get_batch("train")
            logits,loss = model(X,Y)
            losses[k]   = loss.item()   
        out[split] = losses.mean()
    model.train()
    return out

In [39]:

optimizer   = torch.optim.AdamW(model.parameters(),lr=learning_rate)


for iter in range(max_iters):
    if iter % eval_interval == 0:
        losses  = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f},val loss {losses['val']:.4f}")

    #sample a batch of data
    xb,yb = get_batch("train")
    #evaluate the loss
    logits,loss = model(xb,yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()


context = torch.zeros((1,1),dtype=torch.long,device="cuda:0")
print(decode(model.generate(context,max_new_tokens=500)[0]))

step 0: train loss 4.2102,val loss 4.2079
step 300: train loss 2.7947,val loss 2.8114
step 600: train loss 2.6613,val loss 2.6982
step 900: train loss 2.5785,val loss 2.5619
step 1200: train loss 2.5296,val loss 2.5167
step 1500: train loss 2.4829,val loss 2.4894
step 1800: train loss 2.4684,val loss 2.4901
step 2100: train loss 2.4141,val loss 2.4525
step 2400: train loss 2.4547,val loss 2.4418
step 2700: train loss 2.3853,val loss 2.4057
step 3000: train loss 2.4194,val loss 2.4179
step 3300: train loss 2.4191,val loss 2.4017
step 3600: train loss 2.3898,val loss 2.3909
step 3900: train loss 2.3705,val loss 2.3748
step 4200: train loss 2.3517,val loss 2.3279
step 4500: train loss 2.3818,val loss 2.3620
step 4800: train loss 2.3233,val loss 2.2906

My not the wyert I trard awen,
Aund, the lere atho pro tots maalend,
Ifer leould,
I magrdsk toe shy plancenes shold gasuthis therd hern sath bor nostay'd wor bows miest? buho ichoa himers yend nath theay ind darmatl:
Atin's, the cor thillo 

## Adding New parameters

In [132]:
batch_size  = 64     # B 
block_size  = 128    # T
n_emd       = 384    # C 
device      = "cuda" if torch.cuda.is_available() else "cpu"
eval_interval   = 500
learning_rate   = 1e-3
max_iters       = 6000 
eval_iter       = 200
n_layers        = 6 
n_heads         = 6
dropout         = 0.2

vocab_size      = 65 

In [133]:
class Head(nn.Module):
    def __init__(self,head_size):
        super().__init__()
        self.key    = nn.Linear(n_emd,head_size,bias=False)   # key projection
        self.query  = nn.Linear(n_emd,head_size,bias=False)   # query projection
        self.value  = nn.Linear(n_emd,head_size,bias=False)   # value projection 
        self.register_buffer('tril',torch.tril(torch.ones(block_size,block_size)))

        self.dropout    = nn.Dropout(dropout)
    def forward(self,x):
        _,T,C   = x.shape
        k       = self.key(x)   # (B,T,head_size) ==> 4,8,16
        q       = self.query(x) # (B,T,head_size) ==> 4,8,16   
        ## compute attention score(affinities)
        wei     = q @ k.transpose(-2,-1) * C ** -0.5   # (B,T,c) @ (B,C,T) ==> (B,T,T)
        wei     = wei.masked_fill(self.tril[:T,:T] == 0, float('-inf'))
        wei     = F.softmax(wei,dim=-1)
        wei     = self.dropout(wei)
        # perform the weighted aggregation of the value
        v       = self.value(x) # (B,T,head_size) ==> 4,8,16
        out     = wei @ v       # (B,T,T) @ (B,T,head_size) ==> (B,T,head_size) 
        return out 

In [134]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads,head_size):
        super().__init__()
        self.heads  = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj   = nn.Linear(n_emd,n_emd)
        self.dropout= nn.Dropout(dropout)     
    def forward(self,x):
        out = torch.cat([head(x) for head in self.heads],dim=-1)   # concatenate in the channel dimension 
        out = self.dropout(self.proj(out))
        #out = self.dropout(out)
        return out 

In [135]:
class FeedForward(nn.Module):
    def __init__(self,n_emd):
        super().__init__()
        self.net    = nn.Sequential(nn.Linear(n_emd,4 * n_emd),
                                    nn.ReLU(),
                                    nn.Linear(4 * n_emd,n_emd),
                                    nn.Dropout(dropout))
    def forward(self,x):
        return self.net(x)

In [136]:
## residual connection : 
class Block(nn.Module):
    def __init__(self,n_emd,n_heads):
        super().__init__()
        head_size                   = n_emd // n_heads 
        self.self_attention_head    = MultiHeadAttention(n_heads,head_size)
        self.ffwd                   = FeedForward(n_emd)
        self.ln1                    = nn.LayerNorm(n_emd)
        self.ln2                    = nn.LayerNorm(n_emd)
    def forward(self,x):
        x   = x + self.self_attention_head(self.ln1(x)) # adding x is residula connection 
        x   = x + self.ffwd(self.ln2(x))                # adding x is residula connection 
        return x 

In [138]:
class BiGramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for next token from a lookup table 
        self.token_embedding_table      = nn.Embedding(vocab_size,n_emd)
        self.position_embedding_table   = nn.Embedding(block_size,n_emd)
        self.block                      = nn.Sequential(*[Block(n_emd=n_emd,n_heads=n_heads) for _ in range(n_layers)]) 
        # The * is necessary because nn.Sequential expects its modules as separate arguments, not as a single list.
        # Correct (with *): nn.Sequential(module1, module2, module3)This is what *[module1, module2, module3] achieves.
        self.ln_f                       = nn.LayerNorm(n_emd)   # final layer norm
        self.lm_head                    = nn.Linear(n_emd,vocab_size)       
        

    def forward(self,idx,target=None):
        B,T         = idx.shape 
        # index and target are both (B,T) tensor of integers
        tok_emb     = self.token_embedding_table(idx)   # its arrange in the shape of ==================================================================>> (B,T,C)
        pos_emb     = self.position_embedding_table(torch.arange(T,device=device)) # This embdding gives the idea of where word is belong in a sentence=>> (T,C) 
        x           = tok_emb + pos_emb                 # combined representation of token and its position.======Broadcasting apply=>> (B,T,C) + (T,C) == (B,T,C) 
        x           = self.block(x)                     # 
        x           = self.ln_f(x)      
        logits      = self.lm_head(x)                   # for getting token_emb to logits we need linear layer,shape ===================================>> (B,T,vocab_size) 

        if target is None:
            loss    = None
        else:    
            B,T,C   = logits.shape
            logits  = logits.view(B*T,C)    # cross entropy input expectation is (minibatch,C)
            target  = target.view(B*T)      
            loss    = nn.functional.cross_entropy(logits,target)
        return logits,loss
    
    def generate(self,idx,max_new_tokens):
        # idx is (B,T) array of indices in the current context. 
        for _ in range(max_new_tokens):
            idx_cond    = idx[:,-block_size:]                   # crop idx to last block_size tokens
            logits,loss = self.forward(idx_cond)                # Get the prediction ==>  (B,T,C)
            logits      = logits[:,-1,:]                        # focus only on last time step  ==>  (B,C)
            probs       = nn.functional.softmax(logits,dim=-1)  # (B,C)
            # sample from the distribution 
            idx_next    = torch.multinomial(probs,num_samples=1)    # (B,1)
            # append sample index to running sequence 
            idx         = torch.cat([idx,idx_next],dim=1)           # (B,T+1)
        return idx


In [139]:
model = BiGramLanguageModel()
model = model.to("cuda:0")

In [140]:
print(sum(p.numel() for p in model.parameters())/1e6, 'M parameters')

10.739777 M parameters


In [141]:

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ["train","val"]:
        losses = torch.zeros(eval_iter)
        for k in range(eval_iter):
            X,Y         = get_batch("train")
            logits,loss = model(X,Y)
            losses[k]   = loss.item()   
        out[split] = losses.mean()
    model.train()
    return out

In [142]:

optimizer   = torch.optim.AdamW(model.parameters(),lr=learning_rate)


for iter in range(max_iters):
    if iter % eval_interval == 0:
        losses  = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f},val loss {losses['val']:.4f}")

    #sample a batch of data
    xb,yb = get_batch("train")
    #evaluate the loss
    logits,loss = model(xb,yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()


context = torch.zeros((1,1),dtype=torch.long,device="cuda:0")
print(decode(model.generate(context,max_new_tokens=500)[0]))

step 0: train loss 4.3598,val loss 4.3430
step 500: train loss 2.5746,val loss 2.5856
step 1000: train loss 2.4391,val loss 2.4695
step 1500: train loss 2.4144,val loss 2.4170
step 2000: train loss 2.4103,val loss 2.4105
step 2500: train loss 2.3939,val loss 2.3706
step 3000: train loss 2.3351,val loss 2.3891
step 3500: train loss 2.3295,val loss 2.3165
step 4000: train loss 2.2885,val loss 2.3243
step 4500: train loss 2.2847,val loss 2.2941
step 5000: train loss 2.2599,val loss 2.2535
step 5500: train loss 2.2221,val loss 2.2317

Upby dupay E? drsouede thuy shotiieop mis pcim ad Ethm s, l is d thitred bueris; in, io ore k; hite m fthiswov; d wad pe fo y biv. t.
I cwch b SoVt e ouy n ct milpilanesuowbech han th me be tht nos aiat swt ph gh m Ich oumoes hilll hiabind m iel'ofp wqp Mee!icoumasookc?WwyRhffdouwiur f is p oofth meah k iy s'bdfit'wofthabs ilyo ansuoe ny bou b ilih N ns s Mc LaaIiod f, nuouiaSs
Facer!re benan l as l!dyrgotr enoffIe m aop Hak ut ved? y an! t, be elld, E b
ms l

In [143]:

optimizer   = torch.optim.AdamW(model.parameters(),lr=learning_rate)


for iter in range(max_iters):
    if iter % eval_interval == 0:
        losses  = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f},val loss {losses['val']:.4f}")

    #sample a batch of data
    xb,yb = get_batch("train")
    #evaluate the loss
    logits,loss = model(xb,yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()


context = torch.zeros((1,1),dtype=torch.long,device="cuda:0")
print(decode(model.generate(context,max_new_tokens=500)[0]))

step 0: train loss 2.2383,val loss 2.2494
step 500: train loss 2.2423,val loss 2.2207
step 1000: train loss 2.2388,val loss 2.2377
step 1500: train loss 2.2066,val loss 2.2081
step 2000: train loss 2.2127,val loss 2.2022
step 2500: train loss 2.2241,val loss 2.1993
step 3000: train loss 2.2401,val loss 2.2120
step 3500: train loss 2.1648,val loss 2.2087
step 4000: train loss 2.1827,val loss 2.1812
step 4500: train loss 2.1795,val loss 2.1779
step 5000: train loss 2.1767,val loss 2.1615
step 5500: train loss 2.1724,val loss 2.1787

Or, rourw
Iy  gotfDndl IOfETd B
AmthtfTl:lLsgfdosthirMRuisththindIf.d otuchN-Ihenknginanund khalnm?h bGit; th ckMWIt goth
Bd wen, ande, o wlat,
kenosun fAB, had ghe s m, wl lt.ms d Hthr boh'eth Ine! ld gr or
Isd s' pe d on
Thwh greanoosobordidowsd oOomsdkaockck grakey fer d win Inyd, fenouboupreowireneedubutis at bws dge t p ilar,ecttt aererdelar tid t.d fwe, Gnonuddlld ast d dacn berowen h hit de. ho wtoopenkm nygrnge,thd olld.top achImelad, merdede; pofm pe

In [ ]:

optimizer   = torch.optim.AdamW(model.parameters(),lr=learning_rate)


for iter in range(max_iters):
    if iter % eval_interval == 0:
        losses  = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f},val loss {losses['val']:.4f}")

    #sample a batch of data
    xb,yb = get_batch("train")
    #evaluate the loss
    logits,loss = model(xb,yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()


context = torch.zeros((1,1),dtype=torch.long,device="cuda:0")
print(decode(model.generate(context,max_new_tokens=500)[0]))

In [144]:

optimizer   = torch.optim.AdamW(model.parameters(),lr=3e-4)


for iter in range(max_iters):
    if iter % eval_interval == 0:
        losses  = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f},val loss {losses['val']:.4f}")

    #sample a batch of data
    xb,yb = get_batch("train")
    #evaluate the loss
    logits,loss = model(xb,yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()


context = torch.zeros((1,1),dtype=torch.long,device="cuda:0")
print(decode(model.generate(context,max_new_tokens=500)[0]))

step 0: train loss 2.1551,val loss 2.1428
step 500: train loss 2.1087,val loss 2.1068
step 1000: train loss 2.0764,val loss 2.0806
step 1500: train loss 2.0774,val loss 2.0518
step 2000: train loss 2.0729,val loss 2.0831
step 2500: train loss 2.0417,val loss 2.0595
step 3000: train loss 2.0124,val loss 2.0609
step 3500: train loss 2.0263,val loss 2.0347
step 4000: train loss 2.0402,val loss 2.0434
step 4500: train loss 2.0231,val loss 2.0280
step 5000: train loss 2.0147,val loss 2.0301
step 5500: train loss 2.0364,val loss 2.0126

Fir wllecefmvitisith thedseth mmir sl Ekenayre be towamyo by liothe t h'f ff;; sth kbed' s l's IkSysed thy w gasan, qIurd w paty fd my fted, the g wrhed.ncmolothethtim bet h hthacheve Ibln fooavl a m
Fhtennar.thel tie, hup?it;an Mhilsthhthe'ckhidgene lrrad
Tis ofurnfbo.op
bOse
F m tha a yers o hen s mot uly Raruge h sthghe
Ps.tin
Pwe ly IIs d w d, d isthio lerstmanfthd asee, cowt,
FmmevrararVpe?olakdstond atals tethcilak s e,
Ast,
Vgesce st wmy m IlHpen t, s 

In [145]:

optimizer   = torch.optim.AdamW(model.parameters(),lr=3e-4)


for iter in range(max_iters):
    if iter % eval_interval == 0:
        losses  = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f},val loss {losses['val']:.4f}")

    #sample a batch of data
    xb,yb = get_batch("train")
    #evaluate the loss
    logits,loss = model(xb,yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()


context = torch.zeros((1,1),dtype=torch.long,device="cuda:0")
print(decode(model.generate(context,max_new_tokens=500)[0]))

step 0: train loss 2.0411,val loss 2.0411
step 500: train loss 2.0405,val loss 2.0154
step 1000: train loss 2.0421,val loss 2.0283
step 1500: train loss 2.0788,val loss 2.0530
step 2000: train loss 2.0502,val loss 2.0769
step 2500: train loss 2.0565,val loss 2.0515
step 3000: train loss 2.0560,val loss 2.0714
step 3500: train loss 2.0406,val loss 2.0340
step 4000: train loss 2.0169,val loss 2.0374
step 4500: train loss 2.0157,val loss 2.0320
step 5000: train loss 2.0185,val loss 2.0252
step 5500: train loss 2.0487,val loss 2.0495

Jid silftwnnooshdingo mathd t s tonsto Ide b, din t bmeaigexthern
Seshyyinohemybr nctilysto fwhe wnsth!dtrimanolthaidipersthlenog, b, m, dy t fd beloathe cyowht c howueato fshelichto wh thyesu ulto tto tetholaftobe awhasoiohtered,'sth th clalmihed, oL y wiame maishdsood dto'aokishchicidenosth mliminegse re am, ne add mitosbyifiv, h yd llis y, ar t slD f ab ur ht r te p Pme pnswhecmethin, ho ath st hetolihe pt spesasthcennintos may.g yeragringe;e, viyosd ththf

In [146]:

optimizer   = torch.optim.AdamW(model.parameters(),lr=1e-5)


for iter in range(max_iters):
    if iter % eval_interval == 0:
        losses  = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f},val loss {losses['val']:.4f}")

    #sample a batch of data
    xb,yb = get_batch("train")
    #evaluate the loss
    logits,loss = model(xb,yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()


context = torch.zeros((1,1),dtype=torch.long,device="cuda:0")
print(decode(model.generate(context,max_new_tokens=500)[0]))

step 0: train loss 2.0226,val loss 2.0273
step 500: train loss 2.0092,val loss 1.9971
step 1000: train loss 2.0192,val loss 2.0001
step 1500: train loss 2.0237,val loss 2.0031
step 2000: train loss 1.9668,val loss 1.9517
step 2500: train loss 1.9745,val loss 1.9878
step 3000: train loss 1.9798,val loss 1.9798
step 3500: train loss 1.9810,val loss 2.0197
step 4000: train loss 2.0127,val loss 1.9656
step 4500: train loss 1.9658,val loss 1.9712
step 5000: train loss 1.9970,val loss 1.9579
step 5500: train loss 2.0108,val loss 1.9982

Buliver,
Py ae be anedewicter
AfitHefCtyyoll eithind
BrstooHgust t-soo-te Jge, y cheguleso!shy
Ise kntt an pelicomizSath ly,
Sy womafth thbbe we Hth gam fp sse momawe sm Hme thesaloponnelye twowat t:
Shohad yom ulineng.the whimotowhth oneso lit menonoath asteairtohisenenowiboio pr the;n hayour heatholyont measvthisk c sthe,
W feasthethin, thisto lt, t osime the th;ede.we h thofr, t w,
Ne pe, the
'hr, bea; ape t g p Rsnenomalo anosowcothakine ig,
-whe to tria 

In [147]:

optimizer   = torch.optim.AdamW(model.parameters(),lr=1e-6)


for iter in range(max_iters):
    if iter % eval_interval == 0:
        losses  = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f},val loss {losses['val']:.4f}")

    #sample a batch of data
    xb,yb = get_batch("train")
    #evaluate the loss
    logits,loss = model(xb,yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()


context = torch.zeros((1,1),dtype=torch.long,device="cuda:0")
print(decode(model.generate(context,max_new_tokens=500)[0]))

step 0: train loss 1.9717,val loss 1.9716
step 500: train loss 2.0067,val loss 2.0011
step 1000: train loss 2.0083,val loss 1.9796
step 1500: train loss 1.9952,val loss 1.9721
step 2000: train loss 1.9876,val loss 1.9807
step 2500: train loss 1.9630,val loss 1.9480
step 3000: train loss 1.9505,val loss 1.9689
step 3500: train loss 1.9638,val loss 1.9591
step 4000: train loss 1.9770,val loss 1.9394
step 4500: train loss 1.9720,val loss 1.9678
step 5000: train loss 1.9669,val loss 1.9663
step 5500: train loss 1.9762,val loss 1.9954

Cloterchne fomialie s enyevTiene wilomyt pentstil tisthhvesomas bastinr, strsthraboh nonessthy,
O aworen: owon?s hesha d:
d iname thnilyodlld Maw lothorinoheall ayam magele
In fsth.end, anothmyime los's?eshe wes mear oGire p miwarenoo nvibinememerd, t lo, p
B yo thowoth be wowed iond stotitom, m neaa-magyheath, ths'Gvakfbro, th d b; swe we: g doththe isthe kitisifs'wig thne nusoohyowen's-mweane yssthsise, co-at c kimerd, bfape be kp the, ine shth thap d thths